In [1]:
import sys
print(sys.path) 

sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/nas/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.c

In [2]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [4]:
config = {
    "k": 15,
    "method": "sml",
    "bucket_size": 0,
    "dataset_name": "msmarco",
    "mv_type": 'colbertv2-plaid',
    "mode": "disk",
}

In [5]:
def make_config(config, optim="lazy"): # lazy, ltl, naive
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"submodlib.optimizer={optim}",
                f"embedder.mode={config.get('mode', 'mem')}",
                "baseline.distributed_search=False"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [6]:
os.chdir("CMUVERA_IR_ref")

In [7]:
conf = make_config(config)

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=msmarco', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=lazy', 'embedder.mode=disk', 'baseline.distributed_search=False']


In [8]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [9]:
retriever = get_method(conf)
retriever.run()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8841823/8841823 [00:34<00:00, 258484.26it/s]
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: anthropological definition of environment, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1, 28395,  6210,  1997,  4044,   102,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



PermissionError: [Errno 13] Permission denied: './experiments/msmarco/BERT'

In [ ]:
conf = make_config(config, optim="naive")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=naive', 'embedder.mode=disk', 'baseline.distributed_search=False']


In [ ]:
retriever = get_method(conf)
retriever.run()

Naive optimizer is not recommended, proceeding with caution
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 120631.25it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl


Corpus: 2it [00:00,  3.91it/s]
[||||                ]20% [Iteration 3 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 156.20it/s]]6% [Iteration 1 of 15]
Corpus: 2it [00:00,  3.57it/s]
[||||                ]20% [Iteration 3 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 163.60it/s]
Corpus: 2it [00:00,  4.16it/s]
[||||||              ]33% [Iteration 5 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 163.90it/s]6% [Iteration 1 of 15]
Processing queries: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 2519.47it/s]


(tensor([[2716, 1858, 2660, 2022, 2022, 2022],
         [ 816, 2832, 3044, 3044, 3044, 3044],
         [2232, 3462, 3406, 3406, 3406, 3406],
         ...,
         [  52, 1624, 1624, 1624, 1624, 1624],
         [3513, 4486, 2877, 2877, 2877, 2877],
         [4502, 4729, 3080, 1229, 2925, 2925]]),
 tensor([[11.8889, 15.3268, 17.4311, 18.8824, 18.8824, 18.8824],
         [18.3293, 20.9441, 22.3423, 22.3423, 22.3423, 22.3423],
         [15.8303, 18.1585, 20.1529, 20.1529, 20.1529, 20.1529],
         ...,
         [22.5488, 24.3281, 24.3281, 24.3281, 24.3281, 24.3281],
         [21.4312, 24.4695, 25.5001, 25.5001, 25.5001, 25.5001],
         [15.3738, 18.1153, 20.0361, 21.3305, 22.3638, 22.3638]]))

In [ ]:
conf = make_config(config, optim="ltl")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=ltl', 'embedder.mode=disk', 'baseline.distributed_search=False']


In [ ]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 133161.89it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl


Corpus: 2it [00:00,  3.54it/s]
[||||                ]20% [Iteration 3 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 120.15it/s] [Iteration 1 of 15]6% [Iteration 1 of 15]
Corpus: 2it [00:00,  3.80it/s]
[|||||               ]26% [Iteration 4 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 112.87it/s]6% [Iteration 1 of 15] [Iteration 1 of 15]15]15]1 of 15]15]6% [Iteration 1 of 15]                   ]6% [Iteration 1 of 15]
Corpus: 2it [00:00,  3.38it/s]
[||||||              ]33% [Iteration 5 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 121.78it/s]15][Iteration 1 of 15] [Iteration 1 of 15]15]6% [Iteration 1 of 15]15]                   ]6% [Iteration 1 of 15]
Processing queries: 100%|████████████████████████████████████████████████████████

(tensor([[4564, 2716, 2625, 3054, 2022, 2022],
         [ 899, 4814, 3044, 3044, 3044, 3044],
         [2862,  604, 1063,  138,  138,  138],
         ...,
         [2122,  692, 1819, 1624, 1624, 1624],
         [2128, 2152, 4850, 4850, 4850, 4850],
         [2030, 2290,  842,   92, 1608, 1608]]),
 tensor([[11.6709, 13.9376, 15.4270, 16.5174, 17.6283, 17.6283],
         [17.1472, 19.2077, 20.5443, 20.5443, 20.5443, 20.5443],
         [13.0809, 16.4909, 18.8917, 20.9276, 20.9276, 20.9276],
         ...,
         [13.5287, 17.6820, 20.2283, 24.5195, 24.5195, 24.5195],
         [18.9021, 22.8279, 24.2486, 24.2486, 24.2486, 24.2486],
         [14.6383, 17.0231, 18.7446, 19.9932, 21.1217, 21.1217]]))

In [ ]:
with open("./pickles/results/greedy_base_0_128_k15_{config['dataset_name']}_bf.pkl", "rb") as f:
    results = pickle.load(f)

with open("./pickles/results/greedy_submodlib_NaiveGreedy_k15_{config['dataset_name']}_bf_k15.pkl", "rb") as f:
    results_naive = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazyGreedy_k15_{config['dataset_name']}_bf_k15.pkl", "rb") as f:
    results_lazy = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazierThanLazyGreedy_k15_{config['dataset_name']}_bf_k15.pkl", "rb") as f:
    results_ltl = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: "./pickles/results/greedy_base_0_128_k15_{config['dataset_name']}_bf.pkl"

In [ ]:
results_naive[0][0], results_naive[1][0]

(tensor([2716, 1858, 2660, 2022, 2022, 2022]),
 tensor([11.8889, 15.3268, 17.4311, 18.8824, 18.8824, 18.8824]))

In [ ]:
results_lazy[0][0], results_lazy[1][0]

(tensor([2716, 1858, 2660, 2022, 2022, 2022]),
 tensor([11.8889, 15.3268, 17.4311, 18.8824, 18.8824, 18.8824]))

In [ ]:
results_ltl[0][0], results_ltl[1][0]

(tensor([4564, 2716, 2625, 3054, 2022, 2022]),
 tensor([11.6709, 13.9376, 15.4270, 16.5174, 17.6283, 17.6283]))

In [ ]:
results_naive[0][-1], results_naive[1][-1]

(tensor([4502, 4729, 3080, 1229, 2925, 2925]),
 tensor([15.3738, 18.1153, 20.0361, 21.3305, 22.3638, 22.3638]))

In [ ]:
results[0]

[(2716, 11.888933181762695),
 (1858, 15.326838493347168),
 (2660, 17.4311466217041),
 (2022, 18.882389068603516),
 (1853, 19.841398239135742),
 (190, 20.44447898864746),
 (1321, 20.73671531677246),
 (916, 20.99953842163086),
 (4404, 21.165863037109375),
 (1421, 21.295223236083984),
 (1632, 21.38498306274414),
 (2967, 21.44413185119629),
 (1864, 21.495826721191406),
 (4658, 21.539615631103516),
 (2495, 21.570064544677734)]

In [ ]:
# naive, lazy and greedy baseline should all give the same results. ltl may differ.
for item_ind in range(len(results)):
    # Check if naive and greedy baseline results match
    greedy_eles = [x[0] for x in results[item_ind]]
    greedy_scores = [x[1] for x in results[item_ind]]

    res_naive = results_naive[0][item_ind].tolist()
    res_naive_scores = results_naive[1][item_ind].tolist()

    res_lazy = results_lazy[0][item_ind].tolist()
    res_lazy_scores = results_lazy[1][item_ind].tolist()

    assert set(res_naive).issubset(set(greedy_eles)), f"Mismatch at index {item_ind}: {greedy_eles} != {res_naive}"
    # assert set(res_naive_scores).issubset(set(greedy_scores)), f"Mismatch at index {item_ind}: {greedy_scores} != {res_naive_scores}"
    
    # Check if naive and lazy baseline results match
    assert set(res_lazy).issubset(set(greedy_eles)), f"Mismatch at index {item_ind}: {greedy_eles} != {res_lazy}"
    # assert greedy_scores == res_lazy_scores, f"Mismatch at index {item_ind}: {greedy_scores} != {res_lazy_scores}"